# Phase 1 Testing — Model Artefact Validation

This notebook verifies that all saved model artefacts are present, load correctly, and have the expected structure.

**Why test artefacts?**
Before the model can be used — whether in the Flask API or for new predictions — four files must exist in `model_artefacts/`:

| File | Purpose |
|------|---------|
| `lgbm_fraud_model.pkl` | The trained LightGBM classifier |
| `scaler.pkl` | The fitted StandardScaler (must be the same one used during training) |
| `feature_columns.pkl` | The ordered list of all feature column names the model expects |
| `num_features.pkl` | The subset of numeric columns that get scaled |

If any of these are missing or corrupted, every prediction will either error or silently produce wrong results.

**Scope:** No Flask, no API calls, no network. Pure file + object validation.

**How to read results:** Each test prints `✅ PASS` or `❌ FAIL` with a clear message. All 5 must pass before the notebook is safe to submit.

In [2]:
# ── Setup: imports and artefact path ─────────────────────────────────────────
import os
import joblib

ARTEFACT_DIR = os.path.join(os.getcwd(), "model_artefacts")

def check(condition, pass_msg, fail_msg):
    """Simple pass/fail assertion helper."""
    if condition:
        print(f"✅ PASS — {pass_msg}")
    else:
        print(f"❌ FAIL — {fail_msg}")
    return condition

print(f"Artefact directory: {ARTEFACT_DIR}")
check(os.path.isdir(ARTEFACT_DIR),
      "model_artefacts/ directory found",
      f"model_artefacts/ directory NOT found at {ARTEFACT_DIR} — run Section 9 of the main notebook first")

Artefact directory: c:\Users\jolanta.stutane\Desktop\RSU_AI\ML_Final_work\mlbasics_project_nobeigties\model_artefacts
✅ PASS — model_artefacts/ directory found


True

---
## Test 1 — All artefact files exist on disk

Before loading anything, confirm that all four `.pkl` files are physically present in `model_artefacts/`. A missing file means either the notebook's Section 9 save cell was never run, or a file was accidentally deleted.

In [4]:
# Test 1 — All four .pkl files must exist on disk
expected_files = [
    "lgbm_fraud_model.pkl",
    "scaler.pkl",
    "feature_columns.pkl",
    "num_features.pkl",
]

all_present = True
for fname in expected_files:
    path = os.path.join(ARTEFACT_DIR, fname)
    exists = os.path.isfile(path)
    all_present = all_present and exists
    check(exists, f"{fname} found", f"{fname} is MISSING from {ARTEFACT_DIR}")
print()
check(all_present, "All artefact files present", "One or more artefact files are missing")

✅ PASS — lgbm_fraud_model.pkl found
✅ PASS — scaler.pkl found
✅ PASS — feature_columns.pkl found
✅ PASS — num_features.pkl found

✅ PASS — All artefact files present


True

---
## Test 2 — `feature_columns` is a non-empty list of strings

`feature_columns.pkl` stores the exact ordered list of column names the model was trained on. At inference time every transaction must be shaped to match this list exactly — extra columns dropped, missing columns filled with 0.

This test checks: the file loads, it is a `list`, it is not empty, and every entry is a `str`.

In [5]:
# Test 2 — feature_columns must be a non-empty list of strings
feature_cols = joblib.load(os.path.join(ARTEFACT_DIR, "feature_columns.pkl"))

check(isinstance(feature_cols, list),
      "feature_columns loaded as a list",
      f"feature_columns has wrong type: {type(feature_cols)}")

check(len(feature_cols) > 0,
      f"feature_columns is non-empty ({len(feature_cols)} columns)",
      "feature_columns is an empty list")

non_strings = [x for x in feature_cols if not isinstance(x, str)]
check(len(non_strings) == 0,
      "All entries in feature_columns are strings",
      f"Non-string entries found: {non_strings[:5]}")

print(f"\nFirst 5 feature columns : {feature_cols[:5]}")
print(f"Last  5 feature columns : {feature_cols[-5:]}")

✅ PASS — feature_columns loaded as a list
✅ PASS — feature_columns is non-empty (39 columns)
✅ PASS — All entries in feature_columns are strings

First 5 feature columns : ['amt', 'lat', 'long', 'city_pop', 'merch_lat']
Last  5 feature columns : ['state_NM', 'state_OR', 'state_UT', 'state_WA', 'state_WY']


---
## Test 3 — `num_features` is a subset of `feature_columns`

`num_features.pkl` lists the numeric columns that get passed through the StandardScaler. Every column in that list must also appear in `feature_columns` — if they diverge, the scaler will try to transform columns the model doesn't know about, causing a silent data-pipeline mismatch.

In [6]:
# Test 3 — num_features must be a subset of feature_columns
num_features = joblib.load(os.path.join(ARTEFACT_DIR, "num_features.pkl"))

check(isinstance(num_features, list),
      "num_features loaded as a list",
      f"num_features has wrong type: {type(num_features)}")

check(len(num_features) > 0,
      f"num_features is non-empty ({len(num_features)} columns)",
      "num_features is an empty list")

feature_set = set(feature_cols)
orphans = [c for c in num_features if c not in feature_set]
check(len(orphans) == 0,
      "All num_features are present in feature_columns (no pipeline mismatch)",
      f"These num_features are NOT in feature_columns: {orphans}")

print(f"\nNumeric features ({len(num_features)}): {num_features}")

✅ PASS — num_features loaded as a list
✅ PASS — num_features is non-empty (12 columns)
✅ PASS — All num_features are present in feature_columns (no pipeline mismatch)

Numeric features (12): ['amt', 'log_amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'hour', 'day_of_week', 'month', 'age', 'distance_km']


---
## Test 4 — Scaler has a `transform()` method

The scaler must be a properly **fitted** sklearn transformer. An unfitted scaler has no `mean_` attribute. If someone accidentally saved an unfitted scaler, this test will catch it — the scaler would have a `transform()` method but calling it would fail.

In [7]:
# Test 4 — Scaler must be a fitted sklearn transformer
scaler = joblib.load(os.path.join(ARTEFACT_DIR, "scaler.pkl"))

check(hasattr(scaler, "transform"),
      "Scaler has a transform() method",
      "Loaded scaler does not have a transform() method")

check(hasattr(scaler, "mean_"),
      "Scaler is fitted (has mean_ attribute)",
      "Scaler does NOT appear to be fitted — mean_ attribute is missing")

check(len(scaler.mean_) == len(num_features),
      f"Scaler was fitted on {len(scaler.mean_)} features — matches num_features length",
      f"Scaler fitted on {len(scaler.mean_)} features but num_features has {len(num_features)} — mismatch!")

print(f"\nScaler type : {type(scaler).__name__}")
print(f"Fitted on   : {len(scaler.mean_)} numeric features")

✅ PASS — Scaler has a transform() method
✅ PASS — Scaler is fitted (has mean_ attribute)
✅ PASS — Scaler was fitted on 12 features — matches num_features length

Scaler type : StandardScaler
Fitted on   : 12 numeric features


---
## Test 5 — Model has `predict()` and `predict_proba()` methods

The final artefact check: the model must be a proper classifier. `predict()` returns the class label (0 = legit, 1 = fraud). `predict_proba()` returns the probability score used for threshold tuning. Both must be present — the API and the threshold optimisation analysis in Section 6.4 both rely on `predict_proba()`.

In [8]:
# Test 5 — Model must expose predict() and predict_proba()
model = joblib.load(os.path.join(ARTEFACT_DIR, "lgbm_fraud_model.pkl"))

check(hasattr(model, "predict"),
      "Model has a predict() method",
      "Model does NOT have a predict() method")

check(hasattr(model, "predict_proba"),
      "Model has a predict_proba() method",
      "Model does NOT have a predict_proba() method — threshold tuning will fail")

check(hasattr(model, "n_features_in_"),
      f"Model reports n_features_in_ = {getattr(model, 'n_features_in_', '?')}",
      "Model does not expose n_features_in_")

if hasattr(model, "n_features_in_"):
    check(model.n_features_in_ == len(feature_cols),
          f"Model expects {model.n_features_in_} features — matches feature_columns length",
          f"Model expects {model.n_features_in_} features but feature_columns has {len(feature_cols)} — MISMATCH")

print(f"\nModel type  : {type(model).__name__}")
print(f"Features in : {getattr(model, 'n_features_in_', 'unknown')}")

✅ PASS — Model has a predict() method
✅ PASS — Model has a predict_proba() method
✅ PASS — Model reports n_features_in_ = 39
✅ PASS — Model expects 39 features — matches feature_columns length

Model type  : LGBMClassifier
Features in : 39


---
## Phase 1 Summary

If all cells above printed `✅ PASS`, the artefact layer is healthy:
- All four `.pkl` files are on disk
- `feature_columns` and `num_features` are consistent with each other
- The scaler is fitted and aligned to `num_features`
- The model is a valid classifier aligned to `feature_columns`

**Next:** Phase 2 will test the feature engineering logic — verifying that `haversine_approx`, age calculation, `log_amt`, and one-hot encoding all produce correct outputs on known inputs.